# Time Series Analysis in Medicine and Biology
## Practical Course — University of Tübingen · PfeiferLab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamsaraE/time-series-medicine-biology/blob/main/teaching/02_malaria_cases_multiplicative_model.ipynb)

---

# Notebook 02 — Multiplicative Seasonality: Malaria in Kericho

**In this notebook you will:**
1. Recognise **multiplicative** seasonal structure (amplitude grows with the level of the series).
2. See why a **log transform** turns a multiplicative series into an additive one.
3. Compare three decompositions: **classical additive (on log)**, **classical multiplicative**, and **STL**.
4. Fit a simple **log-linear trend** by OLS.

**Dataset:** monthly malaria cases from Kericho, Kenya, with weather covariates
(`Year, Month, Cases, Rain, minT, maxT, VCAP`), loaded from this repository's `data/` folder.

**Why this matters:** infectious-disease counts often swing *proportionally* — a busy season is
not "+X cases" but "×K times" the baseline. Spotting additive vs. multiplicative structure
decides whether you model on the raw or the log scale, which changes every downstream step.

## 1 · Load and prepare the data

We map month names to numbers, build a monthly datetime index, and compute `log_cases`.
A *safe* log is used: any zero count is replaced by 1 before taking the log, so `log(0)` never occurs.

> If the repository link is ever unavailable, the same dataset is mirrored at
> https://sites.google.com/view/tsbiostat/home

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

DATA_URL = "https://raw.githubusercontent.com/ShamsaraE/time-series-medicine-biology/main/data/Kericho_data.csv"
df = pd.read_csv(DATA_URL)

month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
df['Month_num'] = df['Month'].map(month_map)
df['date'] = pd.to_datetime(dict(year=df['Year'], month=df['Month_num'], day=1))
df = df.sort_values('date').set_index('date').asfreq('MS')

df['Cases_adj'] = df['Cases'].replace(0, 1)   # safe log: avoid log(0)
df['log_cases'] = np.log(df['Cases_adj'])

print(f"{len(df)} months, {df.index.min().date()} to {df.index.max().date()}")
df[['Cases', 'log_cases']].head()

## 2 · Raw vs. log series — why the log helps

Look at the two plots below. On the **raw** scale the seasonal swings get *larger* when the
overall number of cases is higher — the hallmark of **multiplicative** structure. On the
**log** scale those swings become roughly **constant in size**, i.e. additive. That is exactly
why we model malaria on the log scale.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df['Cases'], color="firebrick")
ax[0].set_title("Malaria cases — raw scale (amplitude grows with level)")
ax[0].set_ylabel("Cases")
ax[1].plot(df['log_cases'], color="darkgreen")
ax[1].set_title("Malaria cases — log scale (amplitude roughly constant)")
ax[1].set_ylabel("log(cases)"); ax[1].set_xlabel("Year")
plt.tight_layout(); plt.show()

## 3 · Classical additive decomposition (on the log scale)

Because the log series is additive, we can apply an **additive** `seasonal_decompose`.
A multiplicative model on the original scale, $Y_t = T_t \times S_t \times R_t$, becomes additive
after taking logs: $\log Y_t = \log T_t + \log S_t + \log R_t$.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

result_add = seasonal_decompose(df['log_cases'], model='additive', period=12)
result_add.plot()
plt.suptitle("Additive decomposition of log(cases)", y=1.02)
plt.tight_layout(); plt.show()

## 4 · Classical multiplicative decomposition (original scale)

For comparison, `seasonal_decompose` can model the **raw** series directly as
$Y_t = T_t \times S_t \times R_t$. Here the seasonal component is a set of *multipliers* around 1
(e.g. 1.4 = a month 40% above trend) rather than additive offsets.

In [ ]:
result_mult = seasonal_decompose(df['Cases_adj'], model='multiplicative', period=12)
result_mult.plot()
plt.suptitle("Multiplicative decomposition of raw cases", y=1.02)
plt.tight_layout(); plt.show()

## 5 · STL decomposition (flexible, evolving seasonality)

Classical decomposition assumes the seasonal shape is **fixed**. STL
(Seasonal-Trend decomposition using Loess) lets the seasonal pattern **change gradually**
over the years, which is realistic for a long surveillance series where transmission
dynamics shift. We run STL on the log series.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(df['log_cases'], period=12)
res_stl = stl.fit()
res_stl.plot()
plt.suptitle("STL decomposition of log(cases)", y=1.02)
plt.tight_layout(); plt.show()

## 6 · Log-linear trend by OLS

Finally, the simplest possible trend model: regress `log_cases` on time. The slope is the
**average monthly growth rate** on the log scale; exponentiating it gives the multiplicative
factor per month.

In [ ]:
import statsmodels.api as sm

t = np.arange(len(df))
X = sm.add_constant(t)
ols = sm.OLS(df['log_cases'].values, X).fit()

slope = ols.params[1]
print(f"Monthly log-growth slope: {slope:.4f}")
print(f"=> multiplicative factor per month: {np.exp(slope):.4f} "
      f"({(np.exp(slope)-1)*100:+.2f}% per month)")

plt.figure()
plt.plot(df.index, df['log_cases'], label="log(cases)", color="darkgreen", alpha=0.7)
plt.plot(df.index, ols.fittedvalues, label="OLS log-linear trend", color="black", lw=2)
plt.title("Log-linear trend fit"); plt.xlabel("Year"); plt.ylabel("log(cases)"); plt.legend()
plt.tight_layout(); plt.show()

## Key takeaways

- **Growing seasonal amplitude** signals multiplicative structure; a **log transform** converts it to additive so standard additive tools apply.
- Classical decomposition assumes a **fixed** seasonal shape; **STL** allows it to evolve — usually a better fit for long surveillance series.
- A log-linear OLS slope reads directly as an **average growth rate** per period.

**Try it yourself:**
1. Re-run the additive decomposition on the *raw* series and compare the residuals to the log version — which looks more like noise?
2. Add the weather covariates (`Rain`, `minT`, `maxT`) as regressors in the OLS. Do they explain part of the trend?
3. Vary the STL `seasonal` smoothing parameter and watch how rigid vs. flexible the seasonal component becomes.